<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-6-advanced-RAG/week6_advanced_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 Assignment | Advanced Retrieval: Transform and Re-rank
## COSC 650 Applied LLM Systems
### Ben Miller
### Extends Week 5's RAG pipeline (PR #12): https://github.com/BenMillerDev/Applied-LLM-Systems/pull/12

# Part 1: Transform and re-rank
Extend your Week 5 retriever with at least one query transformation and a re-ranking step over the retrieved candidates.

### Rebuild the Week 5 retriever (small (120/20) chunking)

Reusing the same 11-document NailSalonApp corpus and the small (120/20) chunking
config from Week 5, since it performed the worst of the three I tested and had the most room for improvement.

In [1]:
import os, re, pathlib, numpy as np

REPO_URL = "https://github.com/BenMillerDev/Applied-LLM-Systems.git"

if not os.path.exists('docs'):
    # docs/ isn't sitting next to this notebook already -- pull Week 5's docs
    # from main, where the Week 5 PR is already merged
    !rm -rf _repo
    !git clone --quiet {REPO_URL} _repo
    !cp -r _repo/week-5/docs .

TOP_K = 3
CANDIDATE_N = 12  # wider pool retrieved before re-ranking narrows back to TOP_K
DOCS_DIR = pathlib.Path('docs')
DOCS = {p.stem: p.read_text() for p in sorted(DOCS_DIR.glob('*.md'))}

print(f'loaded {len(DOCS)} docs:', list(DOCS.keys()))

loaded 11 docs: ['analytics', 'architecture', 'auth', 'availability', 'client_profiles', 'customer_booking_flow', 'onboarding', 'realtime_sync', 'roadmap', 'service_management', 'time_slot_algorithm']


### Set up the bi-encoder and the cross-encoder reranker

The bi-encoder (`all-MiniLM-L6-v2`, same as Week 5) embeds the query and every
chunk independently, which is fast but can't see the two together. The
`CrossEncoder` reads a query and one candidate chunk as a single pair and
scores them jointly.

In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder

!pip install -q faiss-cpu sentence-transformers
import faiss

os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### Chunking and retrieval functions (from Week 5)

Carried over unchanged from Week 5, simplified to build a single index
instead of a dict of configs.

In [3]:
def chunk_fixed(text, chunk_size, overlap):
    """Split text into fixed-size, overlapping chunks."""
    normalized_text = re.sub(r'\s+', ' ', text).strip()
    chunks = []
    start = 0
    while start < len(normalized_text):
        end = start + chunk_size
        chunks.append(normalized_text[start:end])
        start += chunk_size - overlap
    return chunks

def small_chunker(text):
    """Chunk text into small, overlapping pieces (120 characters, 20-character overlap)."""
    return chunk_fixed(text, 120, 20)

def build_chunks(docs, chunk_fn):
    """
    Chunk each document separately (not one concatenated blob) so a chunk
    never straddles two different docs, and tag each chunk with the doc
    name it came from -- the relevance labeling below needs that source tag.
    """
    all_chunks = []
    chunk_sources = []
    for doc_name, doc_text in docs.items():
        doc_chunks = chunk_fn(doc_text)
        all_chunks.extend(doc_chunks)
        chunk_sources.extend([doc_name] * len(doc_chunks))
    return all_chunks, chunk_sources

def build_index(chunks):
    """Embed a list of chunks and load them into a FAISS index for similarity search."""
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings)
    return index

chunks, sources = build_chunks(DOCS, small_chunker)
index = build_index(chunks)
print(f"small (120/20) chunking (Week 5's worst-recall config) -> {len(chunks)} chunks")

small (120/20) chunking (Week 5's worst-recall config) -> 127 chunks


In [4]:
def retrieve_indices(query, k):
    """Return the positions of the top-k chunks most similar to the query."""
    query_embedding = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, retrieved_positions = index.search(query_embedding, k)
    # FAISS pads with -1 when fewer than k chunks exist in the index -- drop those.
    return [position for position in retrieved_positions[0] if position >= 0]

def retrieve(query, k=TOP_K):
    """Return the actual chunk text for the top-k chunks most similar to the query."""
    return [chunks[i] for i in retrieve_indices(query, k)]

### Query transformation: HyDE (Hypothetical Document Embeddings)

HyDE asks an LLM to write a hypothetical passage that *would* answer the question, in the same style as the docs, and embeds that hypothetical passage instead of the literal question. This costs one LLM call per query.  Its assumption is "the LLM's best guess at the answer is closer in form and vocabulary to the real answer than the raw question is."

In [5]:
if 'google.colab' in str(get_ipython()):
    from google.colab import userdata
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass  # secret not set up in Colab yet -- HyDE falls back to the raw query below

def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live HyDE generation:', LIVE)

live HyDE generation: True


In [6]:
def hyde_generate(query):
    """
    Ask an LLM to write a short hypothetical passage, in the same
    documentation style as the corpus, that would answer the query -- then
    that passage (not the literal question) is what gets embedded and
    searched.
    """
    prompt = (
        'Write a short passage (2-3 sentences) from internal technical '
        'documentation for a nail salon booking app that would answer this '
        'question. Write it as a factual statement, not a question, in a '
        'confident, documentation-like style -- your best guess at the '
        'specific detail, not a hedge.\n\n'
        f'Question: {query}\nPassage:'
    )
    passage = gemini_chat([{'role': 'user', 'content': prompt}], temperature=0)
    return passage if passage is not None else query

### Re-ranking step: cross-encoder

Retrieve a wider candidate pool (`CANDIDATE_N`, not just `TOP_K`) using the
HyDE-generated hypothetical passage, so there's room for the cross-encoder to
promote a correct chunk the bi-encoder ranked outside the top 3. Reranking scores each candidate against the ORIGINAL query, not the
hypothetical passage.

In [7]:
def rerank(query, candidate_indices, k=TOP_K):
    """
    Score each candidate chunk against the original query with a
    cross-encoder, and return the top-k candidate indices by that score.
    """
    pairs = [(query, chunks[i]) for i in candidate_indices]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidate_indices, scores), key=lambda pair: pair[1], reverse=True)
    return [index for index, score in ranked[:k]]

def transformed_retrieve(query, k=TOP_K, candidate_n=CANDIDATE_N, hypothetical_passage=None):
    """
    Week 6's retrieval: retrieve a wide candidate pool using a HyDE
    hypothetical passage's embedding, then re-rank that pool against the
    ORIGINAL query with a cross-encoder to narrow back down to k.
    """
    if hypothetical_passage is None:
        hypothetical_passage = hyde_generate(query)
    candidate_indices = retrieve_indices(hypothetical_passage, candidate_n)
    return rerank(query, candidate_indices, k)

# Part 2: Measure the effect
Run the same query set as Week 5 and report retrieval quality before and after, with per-query deltas. State which query types benefited and which did not.

### Evaluation functions (from Week 5)

In [8]:
def label_relevant(source_doc, fact):
    """
    Find every chunk that counts as ground-truth relevant for one query: it
    has to come from the right document AND actually contain the fact's
    wording.
    """
    fact_lower = fact.lower()
    return [i for i, (chunk_text, chunk_source) in enumerate(zip(chunks, sources))
            if chunk_source == source_doc and fact_lower in chunk_text.lower()]

def precision_recall(retrieved_indices, source_doc, fact):
    """
    Precision is how many of the retrieved chunks were actually relevant;
    recall is how many of the relevant chunks that exist were actually
    retrieved.
    """
    relevant_indices = label_relevant(source_doc, fact)
    hits = [i for i in retrieved_indices if i in relevant_indices]
    precision = len(hits) / len(retrieved_indices) if retrieved_indices else 0.0
    recall = len(hits) / len(relevant_indices) if relevant_indices else 0.0
    return precision, recall

### Same 11 queries as Week 5

In [9]:
# Query, the fact that answers it, and which doc that fact lives in -- identical to Week 5
qa = [
    ('How far in advance must a customer book?', '1 hour', 'time_slot_algorithm'),
    ('What does the owner app use for navigation?', 'Expo Router', 'architecture'),
    ('How do new bookings appear on the dashboard without a refresh?', 'no pull-to-refresh or manual reload needed', 'realtime_sync'),
    ("Where do new owners' service lists start from?", 'default service catalog', 'service_management'),
    ('How many steps are in the customer booking flow?', 'four steps', 'customer_booking_flow'),
    ('What happens to Google sign-in when running in Expo Go?', 'mock version', 'auth'),
    ('What three time ranges can the analytics dashboard filter by?', 'This Week, This Month, and This Year', 'analytics'),
    ('What two fields track onboarding progress?', "dismissed the checklist", 'onboarding'),
    ("How is a client's fill due date determined?", 'computed fill due date', 'client_profiles'),
    ('What limits how many bookings an owner can take in a day?', 'maximum number of bookings allowed per day', 'availability'),
    ('Does the app support SMS booking confirmations today?', 'SMS booking confirmations', 'roadmap'),
]

### Before vs. after, per query

In [10]:
print(f'{"query":65s} {"precision":>22s} {"recall":>18s}')
results = []
for query, fact, source_doc in qa:
    baseline_indices = retrieve_indices(query, TOP_K)
    baseline_precision, baseline_recall = precision_recall(baseline_indices, source_doc, fact)

    # Generate the HyDE passage ONCE per query and reuse it for retrieval
    hypothetical_passage = hyde_generate(query)
    transformed_indices = transformed_retrieve(query, hypothetical_passage=hypothetical_passage)
    transformed_precision, transformed_recall = precision_recall(transformed_indices, source_doc, fact)

    results.append({
        'query': query, 'fact': fact, 'source_doc': source_doc,
        'hypothetical_passage': hypothetical_passage,
        'baseline': (baseline_precision, baseline_recall),
        'transformed': (transformed_precision, transformed_recall),
    })

    precision_delta = transformed_precision - baseline_precision
    recall_delta = transformed_recall - baseline_recall
    flag = 'REGRESSED' if recall_delta < 0 else ('IMPROVED' if (precision_delta > 0 or recall_delta > 0) else 'same')
    print(f'{query[:63]:65s} P {baseline_precision:.2f}->{transformed_precision:.2f} '
          f'({precision_delta:+.2f})   R {baseline_recall:.2f}->{transformed_recall:.2f} '
          f'({recall_delta:+.2f})   {flag}')

mean_baseline_p = np.mean([r['baseline'][0] for r in results])
mean_baseline_r = np.mean([r['baseline'][1] for r in results])
mean_transformed_p = np.mean([r['transformed'][0] for r in results])
mean_transformed_r = np.mean([r['transformed'][1] for r in results])
print()
print(f'mean precision@{TOP_K}: {mean_baseline_p:.2f} -> {mean_transformed_p:.2f}')
print(f'mean recall@{TOP_K}:    {mean_baseline_r:.2f} -> {mean_transformed_r:.2f}')

query                                                                          precision             recall
How far in advance must a customer book?                          P 0.33->0.33 (+0.00)   R 1.00->1.00 (+0.00)   same
What does the owner app use for navigation?                       P 0.33->0.00 (-0.33)   R 1.00->0.00 (-1.00)   REGRESSED
How do new bookings appear on the dashboard without a refresh?    P 0.00->0.33 (+0.33)   R 0.00->1.00 (+1.00)   IMPROVED
Where do new owners' service lists start from?                    P 0.33->0.33 (+0.00)   R 1.00->1.00 (+0.00)   same
How many steps are in the customer booking flow?                  P 0.33->0.33 (+0.00)   R 1.00->1.00 (+0.00)   same
What happens to Google sign-in when running in Expo Go?           P 0.33->0.33 (+0.00)   R 1.00->1.00 (+0.00)   same
What three time ranges can the analytics dashboard filter by?     P 0.00->0.00 (+0.00)   R 0.00->0.00 (+0.00)   same
What two fields track onboarding progress?                      

### Part 2 Results

| | Mean precision@3 | Mean recall@3 |
|---|---|---|
| Baseline (small chunking) | 0.21 | 0.64 |
| HyDE + rerank | 0.24 | 0.73 |

Grouping the eleven queries by type shows three distinct patterns.

**Queries with distinctive terminology already in the docs stayed unchanged.**
Six queries already hit recall@3 = 1.00 at baseline and didn't move either
direction: "How far in advance must a customer book?", "Where do new
owners' service lists start from?", "How many steps are in the customer
booking flow?", "What happens to Google sign-in when running in Expo Go?",
"How is a client's fill due date determined?", and "What limits how many
bookings an owner can take in a day?". Each one contains a specific term
that also appears in the matching doc almost verbatim -- "Expo Go,"
"service catalog," "fill due date" -- so the bi-encoder already finds the
right chunk without help. HyDE doesn't change anything here because there's
nothing to fix.

**Queries phrased as questions about behavior, where the doc states the
fact as a plain statement, improved.** "How do new bookings appear on the
dashboard without a refresh?" and "Does the app support SMS booking
confirmations today?" both went from a complete miss (precision 0.00,
recall 0.00) to precision 0.33, recall 1.00. Both ask about a behavior or
status in question form, while the docs describe the matching fact as a
declarative sentence. HyDE's hypothetical passage is itself a declarative
sentence, so it closes that gap.  This is the case HyDE is built for.

**Queries asking for an enumerated list inside a longer sentence stayed
missed.** "What three time ranges can the analytics dashboard filter by?"
and "What two fields track onboarding progress?" missed at 0.00/0.00 both
before and after. Both ask for a short list of items embedded inside a
longer sentence, and with 120-character chunking the fixed-size boundary
can cut that sentence away from the context that identifies it as the
answer. That's a chunking problem and not a phrasing problem.  Rewording the query with HyDE can't recover an answer that the chunk boundaries split apart.

**One query with an ambiguous term regressed.** "What does the owner app
use for navigation?" dropped from precision 0.33, recall 1.00 to precision
0.00, recall 0.00 -- see Part 3.

Net effect: the whole gain comes from closing the question/statement gap on
two queries. HyDE neither helps nor hurts queries that already matched on
distinctive terminology, can't fix queries where the chunk boundary cuts
off the answer, and actively hurt the one query where it picked the wrong
sense of an ambiguous word.


# Part 3: Find one failure and explain it
Show at least one query where the transformation made retrieval worse, and explain why the technique misfired, since a transformation is an assumption about the query and assumptions break. If no query regressed, report that result and explain why the transformation held across your set.

### Find the worst regression (or confirm there isn't one)

Picks out whichever query in Part 2's results got worse after the
transform + rerank pipeline. Falls back to reporting "no regression" if every query held or improved.

In [11]:
def find_worst_regression(regressions):
    """
    Given a list of regressed queries (each a dict with 'baseline' and
    'transformed' precision/recall tuples), returns the single worst one.

    "Worst" means: biggest drop in recall. If two queries dropped recall by
    the same amount, whichever one also dropped precision more wins the tie.
    """
    worst = regressions[0]
    worst_baseline_precision, worst_baseline_recall = worst['baseline']
    worst_transformed_precision, worst_transformed_recall = worst['transformed']
    worst_recall_drop = worst_transformed_recall - worst_baseline_recall
    worst_precision_drop = worst_transformed_precision - worst_baseline_precision

    for candidate in regressions[1:]:
        baseline_precision, baseline_recall = candidate['baseline']
        transformed_precision, transformed_recall = candidate['transformed']
        recall_drop = transformed_recall - baseline_recall
        precision_drop = transformed_precision - baseline_precision

        if recall_drop < worst_recall_drop:
            worst, worst_recall_drop, worst_precision_drop = candidate, recall_drop, precision_drop
        elif recall_drop == worst_recall_drop and precision_drop < worst_precision_drop:
            worst, worst_recall_drop, worst_precision_drop = candidate, recall_drop, precision_drop

    return worst

In [12]:
import textwrap

# A query counts as a regression if either:
#   (a) recall dropped
#   (b) recall stayed tied but precision dropped
regressions = [r for r in results
    if (r['transformed'][1] - r['baseline'][1]) < 0
    or (r['transformed'][1] == r['baseline'][1] and r['transformed'][0] < r['baseline'][0])
]

if regressions:
    worst = find_worst_regression(regressions)
    query, fact, source_doc = worst['query'], worst['fact'], worst['source_doc']

    print(f'Regressed query: "{query}"')
    print(f'Baseline:    precision={worst["baseline"][0]:.2f} recall={worst["baseline"][1]:.2f}')
    print(f'Transformed: precision={worst["transformed"][0]:.2f} recall={worst["transformed"][1]:.2f}')
    print()

    # Reuse the exact passage generated in Part 2's loop for this query
    hypothetical_passage = worst['hypothetical_passage']
    baseline_indices = retrieve_indices(query, TOP_K)
    transformed_indices = transformed_retrieve(query, hypothetical_passage=hypothetical_passage)
    relevant_indices = label_relevant(source_doc, fact)

    print('=' * 70)
    print('HYPOTHETICAL PASSAGE (generated by HyDE, embedded instead of the raw query)')
    print('=' * 70)
    print(textwrap.fill(hypothetical_passage, width=70))
    print('=' * 70)
    print()

    print('Baseline retrieved:')
    for i in baseline_indices:
        flag = 'relevant' if i in relevant_indices else 'wrong'
        print(f'  [{flag:8s}] ({sources[i]:20s}) "{chunks[i][:90]}..."')
    print()
    print('Transformed (HyDE + rerank) retrieved instead:')
    for i in transformed_indices:
        flag = 'relevant' if i in relevant_indices else 'wrong'
        print(f'  [{flag:8s}] ({sources[i]:20s}) "{chunks[i][:90]}..."')
else:
    print('No query regressed: every query that succeeded at baseline still succeeded '
          'after transform + rerank.')

    print(f'mean precision@{TOP_K}: {mean_baseline_p:.2f} -> {mean_transformed_p:.2f}')
    print(f'mean recall@{TOP_K}:    {mean_baseline_r:.2f} -> {mean_transformed_r:.2f}')

Regressed query: "What does the owner app use for navigation?"
Baseline:    precision=0.33 recall=1.00
Transformed: precision=0.00 recall=0.00

HYPOTHETICAL PASSAGE (generated by HyDE, embedded instead of the raw query)
The owner application utilizes Apple MapKit for iOS and Google Maps
SDK for Android to render all in-app navigation, route optimization,
and location-based scheduling features. These embedded mapping
services ensure real-time traffic integration and precise geolocation
tracking for mobile nail technicians providing on-site services.

Baseline retrieved:
  [wrong   ] (architecture        ) "nicians, split into two separate codebases that serve two different audiences. The owner-f..."
  [relevant] (architecture        ) "Redux. Navigation on the owner app uses Expo Router's file-based routing, with route group..."
  [wrong   ] (analytics           ) "intments collection everything else in the app reads from. The dashboard lets an owner cha..."

Transformed (HyDE + rerank)

### The regression: "What does the owner app use for navigation?"

Baseline got this one right: the correct chunk ("...Navigation on the owner
app uses Expo Router's file-based routing, with route group...") was
retrieved at rank 2 of 3 (precision 0.33, recall 1.00). After the transform,
it's gone -- precision 0.00, recall 0.00.

The cause is the wording of the generated hypothetical passage:

> "The owner application utilizes Apple MapKit for iOS and Google Maps SDK
> for Android to render all in-app navigation, route optimization, and
> location-based scheduling features."

The word "navigation" in the query is ambiguous.  My real documents refer to navigation, meaning in-app screen routing (Expo Router, file-based routing, route groups).  The LLM-generated answer refers to real-world navigation between physical locations. The model picked the wrong interpretation.  It stated a specific, but fictitious
technical detail (there is no Apple MapKit or in-app route optimzation system
anywhere in the real docs).  It also didn't hedge, it gave a confident answer instead of writing something general enough to stay safely ambiguous.

The incorrect interpretation changed which chunks were considered.  `transformed_retrieve` embeds
the hypothetical passage and uses it to pull a pool of 12 candidates
*before* reranking; the cross-encoder only reorders whatever is in that
pool against the original query.  Because the hypothetical passage is about
physical navigation and routes, it pulled in candidates about the time-slot utility module and a real-time sync explanation.  The correct chunk was not in the candidate pool, so reranking could not recover it.